# Лабораторная работа 4

**Тема:** сравнение моделей машинного обучения для задачи регрессии.

В работе используется датасет `ipc_dataset.csv` с индексами потребительских цен. Целью является предсказание четырёх целевых признаков: `total`, `food`, `non_food`, `services`.

## Импорт библиотек и загрузка данных

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import plot_tree

PROJECT_ROOT = Path.cwd().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from data.data_processing import prepare_data
from models.decision_tree_model import get_model as get_tree_model
from models.linear_regression_model import get_model as get_linear_model
from models.svm_kernel_model import get_model as get_svm_kernel_model

In [ ]:
raw_df = pd.read_csv(PROJECT_ROOT / 'data' / 'ipc_dataset.csv', sep=';')
raw_df.head()

## Предобработка данных

В подготовке данных выполняются следующие шаги:

- названия месяцев переводятся в числовой формат;
- строковые числовые значения приводятся к типу float;
- формируются лаговые признаки (`lag1`, `lag2`, `lag3`);
- категориальный признак месяца кодируется через one-hot encoding;
- строки с пропусками после формирования лагов удаляются.

In [4]:
X, y = prepare_data()
print('Размер X:', X.shape)
print('Размер y:', y.shape)
X.head()

Размер X: (834, 24)
Размер y: (834, 4)


,m_1,m_2,m_3,m_4,m_5,m_6,m_7,m_8,m_9,m_10,...,total_lag3,food_lag1,food_lag2,food_lag3,non_food_lag1,non_food_lag2,non_food_lag3,services_lag1,services_lag2,services_lag3
3,False,False,False,True,False,False,False,False,False,False,...,106.2,105.2,103.1,104.6,104.6,105.9,109.0,105.5,105.2,103.2
4,False,False,False,False,True,False,False,False,False,False,...,104.8,171.8,105.2,103.1,170.5,104.6,105.9,121.5,105.5,105.2
5,False,False,False,False,False,True,False,False,False,False,...,106.3,100.5,171.8,105.2,105.4,170.5,104.6,103.9,121.5,105.5
6,False,False,False,False,False,False,True,False,False,False,...,163.5,99.7,100.5,171.8,102.2,105.4,170.5,103.5,103.9,121.5
7,False,False,False,False,False,False,False,True,False,False,...,103.0,97.2,99.7,100.5,102.8,102.2,105.4,104.4,103.5,103.9


## Разделение выборки и обучение моделей

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Decision Tree': get_tree_model(X_train_scaled, y_train),
    'Linear Regression': get_linear_model(X_train_scaled, y_train),
    'SVM': get_svm_kernel_model(X_train_scaled, y_train),
}

rows = []
for model_name, model in models.items():
    predictions = model.predict(X_test_scaled)
    rows.append({
        'Model': model_name,
        'MAE': mean_absolute_error(y_test, predictions),
        'R2': r2_score(y_test, predictions),
    })

results = pd.DataFrame(rows).sort_values(by='R2', ascending=False)
results

## Итоговое сравнение моделей

По результатам эксперимента лучшей моделью стала **линейная регрессия**.

- Linear Regression: `MAE = 27.679629`, `R2 = 0.726351`
- SVM: `MAE = 39.954708`, `R2 = 0.442596`
- Decision Tree: `MAE = 43.150084`, `R2 = 0.375073`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(results['Model'], results['MAE'], color=['#2a9d8f', '#e9c46a', '#e76f51'])
axes[0].set_title('Сравнение моделей по MAE')
axes[0].set_ylabel('MAE')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(results['Model'], results['R2'], color=['#264653', '#8ab17d', '#f4a261'])
axes[1].set_title('Сравнение моделей по R2')
axes[1].set_ylabel('R2')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

![Сравнение моделей](model_comparison.png)

## Важность признаков дерева решений

In [ ]:
tree_model = models['Decision Tree']
feature_importance = pd.Series(tree_model.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(10, 8))
feature_importance.plot(kind='barh', color='#2a9d8f')
plt.title('Важность признаков дерева решений')
plt.xlabel('Важность')
plt.tight_layout()
plt.show()

![Важность признаков](feature_importance.png)

## Визуализация дерева решений

In [ ]:
plt.figure(figsize=(24, 12))
plot_tree(
    tree_model,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3,
)
plt.title('Дерево решений (первые 4 уровня)')
plt.show()

![Дерево решений](decision_tree.png)

## Вывод

Для выбранного набора данных задача была решена как задача многовыходной регрессии. Были обучены три модели: линейная регрессия, SVM и дерево решений. По метрикам `MAE` и `R2` наилучший результат показала линейная регрессия. Дополнительно построены график важности признаков и визуализация дерева решений.